In [1]:
#Ryan Wong
#CPSC 236 - Selected Languages: Python
#Final Project - Quiz Maker (Part 1)

import sys, csv, random, time

filename = "testbank.txt"

def validate_id():
    """
    The user is prompted to enter their firstname, lastname, and student id. Function also validates if the entered student id
    at least six characters long, begins with 'A', and only includes digits between 1 and 9. The user only has 3 attempts to
    enter a valid id, once all attempts are used the program exits
    """
    firstname = input("Enter your first name: ").strip()
    lastname = input("Enter your last name: ").strip()
    
    attempts = 0
    #Only allows a maximum of 3 attempts to enter a valid ID
    while attempts < 3:
        student_id = input("Enter your Student ID: ").strip()
        print()

        #Checking if the length of ID is exactly 6 and starts with 'A"
        if len(student_id) == 6 and student_id.startswith('A'):
            valid_id = True
            #Verify if the remaining 5 characters are digits between 1 and 9
            for char in student_id[1:]:
                if char < '1' or char > '9':
                    valid_id = False
                    break

            #If all checks pass, return the student information
            if valid_id:
                return student_id, firstname, lastname
        
        attempts += 1
        print("Invalid ID format. You have " + str(3-attempts) + " attempt(s) left.")

    #Terminates the program if the maximum number of attempts is reached
    print("You have reached the maximum number of attempts. Exiting program.")
    sys.exit()

def get_questions(filename):
    """
    Reads the testbank.txt file and extracts the multiple choice questions, along with answer options 
    and the answer to that question ('A', 'B', or 'C')
    """
    questions = []
    try:
        #Open CSV file in read mode
        with open(filename, 'r') as file:
            reader = csv.reader(file)
            for row in reader:
                #Ensure the row has enough columns to represent a full question
                if len(row) >= 5:
                    q_dict = {
                        "question": row[0],
                        "A": row[1],
                        "B": row[2],
                        "C": row[3],
                        "answer": row[4].strip().upper()
                    }
                    questions.append(q_dict)
    except FileNotFoundError:
        print("Error: The file '" + filename + "' was not found.")
    except Exception as e:
        print("An unexpected error occurred while reading the file: " + str(e))

    return questions

def run_quiz(student_info, questions):
    """
    Starts up the quiz, which also enforces a 10 minute time limit to finish the quiz. Prompts the user to select 
    either 10 or 20 questions, with the points value adjusting accordingly, student's answers, and generates a formatted 
    results text file upon completion. 
    """
    student_id, firstname, lastname = student_info
    points = 1.0

    #Keeps prompting until the user selects a valid quiz length
    while True:
        choice = int(input("Choose the number of questions (10 or 20): "))
        if choice == 10 or choice == 20:
            num_questions = choice
            break
        print("Invalid choice. Please enter 10 or 20.")

    #Point value is determined based on the length of the quiz
    if num_questions == 10:
        points = 1.0
    else:
        points = 0.5

    #Prevents error if the testbank is too small
    if len(questions) < num_questions:
        print("There isn't enough questions in the testbank.")
        sys.exit()

    #Randomly select the required number of unique questions for the bank
    selected_questions = random.sample(questions, num_questions)

    score = 0.0
    start_time = time.time()
    time_limit = 10 * 60 #10 minutes converted to seconds

    report_data = []

    #Loops through and display each selected question
    for i, q in enumerate(selected_questions):
        print("\nQuestion " + str(i+1) + ": " + q["question"])
        print("A) " + q["A"])
        print("B) " + q["B"])

        #Checks if option C exists: adjust valid answer options accordingly
        if q["C"].strip() != "":
            print("C) " + q["C"])
            options = ['A', 'B', 'C']
        else:
            options = ['A', 'B']

        #Keep prompting untl user provides an appropriate answre from the avaliable options
        while True:
            student_answer = input("Enter your answer: ").strip().upper()
            if student_answer in options:
                break
            if len(options) == 2:
                print("Invalid choice. Choose either A or B.")
            else:
                print("Invalid choice. Choose either A, B or C.")

        #Enforces the 10 minute time limit immediately after an answer is given
        #After 10 minutes when you enter an answer the program closes
        current_time = time.time()
        elapsed = current_time - start_time
        if elapsed > time_limit:
            print("Time is up. 10 minutes have passed. Closing quiz.")
            break

        #Check answer, if correct award points
        correct_answer = q["answer"]
        if student_answer == correct_answer:
            score += points

        #Store the quiz interaction, will be used to write report file later
        report_data.append({
            'q_text': q["question"],
            'correct': correct_answer,
            'student': student_answer
        })

    #Calculate total time elapsed into minutes and seconds
    total_elapsed_time = time.time() - start_time
    minutes = int(total_elapsed_time // 60)
    seconds = int(total_elapsed_time % 60)

    print("\nYou have finished this quiz. Your score: ", score, "/10")

    #Generate the formatted text file with the student's results
    filename = student_id + "_" + firstname + "_" + lastname + ".txt"
    with open(filename, 'w') as file:
        file.write("Student ID: " + student_id + "\n")
        file.write("Name: " + firstname + " " + lastname + "\n")
        file.write("Score: " + str(score) + "/10\n")
        file.write("Elapsed time: " + str(minutes) + " minutes, " + str(seconds) + " seconds\n\n")

        #Output the breakdown of each question into the file
        for item in report_data:
            file.write("Question: " + item['q_text'] + "\n")
            file.write("Correct Answer: " + item['correct'] + "\n")
            file.write("Student's Answer: " + item['student'] + "\n\n")

    print("Results saved to " + filename)
    
def main():
    """
    The main control loop for the quiz program. Handling id validation, fetching the testbank file, running the quiz
    and towards the end of the program after getting the quiz score, it prompts the user to either start a new quiz
    for another student or exit. 
    """
    while True:
        student_info = validate_id()
        questions = get_questions(filename)
        run_quiz(student_info, questions)
        print("-" * 30)
        choice = input("Enter 'S' to start a new quiz for another student, or 'Q' to exit:").strip().upper()
        print()

        #Handle replay/exit logic
        if choice == 'Q':
            print("Exiting program")
            break
        elif choice == 'S':
            print("\n" * 50)
            continue
        else:
            print("Invalid input. Enter 'S' or 'Q'.")
        
if __name__ == '__main__':
    main()

Enter your first name:  crondolius
Enter your last name:  dinkle
Enter your Student ID:  hedontgoton



Invalid ID format. You have 2 attempt(s) left.


Enter your Student ID:  A00023



Invalid ID format. You have 1 attempt(s) left.


Enter your Student ID:  A99999


...


Question 10: What does if statement do?
A) Avoids code block
B) Executes code block
C) Returns code block


Enter your answer:  C



You have finished this quiz. Your score:  5.0 /10
Results saved to A99999_crondolius_dinkle.txt
------------------------------


Enter 'S' to start a new quiz for another student, or 'Q' to exit: Q



Exiting program
